In [0]:
%python

import sys
import os 

sys.path.append(os.path.abspath('..'))

from utils.utils_merge_into_tables import upsert_data

In [0]:
%python
catalog_olist = dbutils.widgets.get("catalog")
schema_gold = dbutils.widgets.get("schema_gold")
output_table = dbutils.widgets.get("output_table")
sk_table_olist = dbutils.widgets.get('sk_table')



In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_silver_customer AS 
SELECT
*
FROM ${catalog}.${schema_silver}.${table_silver_olist_customers};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_silver_orders AS 
SELECT
*
FROM ${catalog}.${schema_silver}.${table_silver_orders};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_silver_orders_items AS 
SELECT
*
FROM ${catalog}.${schema_silver}.${table_silver_order_items};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_dim_customer AS 
SELECT
*
FROM ${catalog}.${schema_gold}.${table_dim_customer};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_dim_products AS 
SELECT
*
FROM ${catalog}.${schema_gold}.${table_dim_products};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_dim_tempo AS
SELECT
*
FROM ${catalog}.${schema_gold}.${table_dim_tempo};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW fct_sales_items AS 
SELECT
XXHASH64(orders.order_id,orders_items.order_item_id) AS SK_SALES_ITEMS,
dim_customer.SK_CUSTOMER,
dim_products.SK_PRODUCT,
dim_tempo.SK_TEMPO,
orders.order_id AS ID_ORDER,
orders_items.order_item_id AS ID_ORDER_ITEM,
DATE(orders.order_purchase_timestamp) AS DT_COMPRA,
DATE_FORMAT(orders.order_purchase_timestamp, 'HH:mm') AS HR_COMPRA,
DATE(orders.order_approved_at) AS DT_PAGAMENTO_APROVADO,
DATE_FORMAT(orders.order_approved_at, 'HH:mm') AS HR_PAGAMENTO_APROVADO,
orders.order_delivered_carrier_date AS DT_ENTREGA_TRANSPORTADORA,
orders.order_delivered_customer_date AS DT_ENTREGA_CLIENTE,
orders.order_estimated_delivery_date AS DT_ENTREGA_ESTIMADA,
orders_items.price AS PRECO,
orders_items.freight_value AS CUSTO_FRETE,
CAST((UNIX_TIMESTAMP(orders.order_approved_at) - UNIX_TIMESTAMP(orders.order_purchase_timestamp)) / 60 AS INT) AS TEMPO_PAGAMENTO_APROVADO_MINUTOS,
DATEDIFF(orders.order_delivered_customer_date, orders.order_purchase_timestamp) AS TEMPO_ENTREGA_CLIENTE_DIAS,
DATEDIFF(orders.order_delivered_customer_date, orders.order_estimated_delivery_date) AS DIAS_ATRASO_ENTREGA,
CASE 
WHEN DATEDIFF(orders.order_delivered_customer_date, orders.order_estimated_delivery_date) > 0 THEN 'ATRASADO'
WHEN DATEDIFF(orders.order_delivered_customer_date, orders.order_estimated_delivery_date) = 0 THEN 'NO PRAZO'
WHEN DATEDIFF(orders.order_delivered_customer_date, orders.order_estimated_delivery_date) < 0 THEN 'ADIANTADO'
END AS STATUS_ATRASO
FROM table_silver_orders orders
LEFT JOIN table_silver_orders_items orders_items
ON orders.order_id = orders_items.order_id
LEFT JOIN table_silver_customer tb_customer_silver
ON orders.customer_id = tb_customer_silver.customer_id
LEFT JOIN table_dim_customer dim_customer
ON tb_customer_silver.customer_unique_id = dim_customer.ID_CUSTOMER
LEFT JOIN table_dim_products dim_products
ON orders_items.product_id = dim_products.ID_PRODUCT
LEFT JOIN table_dim_tempo dim_tempo
ON DATE(orders.order_purchase_timestamp) = dim_tempo.DATE_ACTUAL;



## Merge Table

In [0]:
%run ../setup/00_aws_connection

In [0]:
%python

df_fct_sales_items = spark.table('fct_sales_items')
full_table_name = f"{catalog_olist}.{schema_gold}.{output_table}"

upsert_data(df_fct_sales_items, full_table_name, sk_table_olist,name_bucket,layer='gold')
